In [7]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [8]:
from pathlib import Path
import json
import random

import yaml
from ultralytics import YOLO


In [9]:
SEED = 42
DATASET_ROOT = Path("/kaggle/input/datasets/matweyisupov/self-driving-card")
if not (DATASET_ROOT / "export" / "images").exists():
    DATASET_ROOT = DATASET_ROOT / "Self-Driving-Car-3"

IMAGES_DIR = DATASET_ROOT / "export" / "images"
LABELS_DIR = DATASET_ROOT / "export" / "labels"

WORKDIR = Path("/kaggle/working")
SPLIT_DIR = WORKDIR / "splits" / "self_driving_car"
DATA_YAML = SPLIT_DIR / "data.yaml"

MODEL = "yolov8n.pt"
EPOCHS = 30
IMGSZ = 640
BATCH = 64
DEVICES = [0, 1]
PROJECT = WORKDIR / "runs"
NAME = "yolov8n_baseline"


In [10]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.device_count())
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print(torch.cuda.get_device_capability(0))
print(torch.cuda.get_arch_list())


2.10.0+cu128
12.8
2
['Tesla T4', 'Tesla T4']
(7, 5)
['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [11]:
with open(DATASET_ROOT / "data.yaml", "r", encoding="utf-8") as f:
    meta = yaml.safe_load(f)

image_paths = sorted([p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
random.Random(SEED).shuffle(image_paths)

n = len(image_paths)
n_train = int(n * 0.7)
n_val = int(n * 0.2)

splits = {
    "train": image_paths[:n_train],
    "val": image_paths[n_train:n_train + n_val],
    "test": image_paths[n_train + n_val:],
}

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, split_images in splits.items():
    (SPLIT_DIR / f"{split_name}.txt").write_text("\n".join(str(p) for p in split_images) + "\n", encoding="utf-8")

data_config = {
    "path": str(DATASET_ROOT),
    "train": str((SPLIT_DIR / "train.txt").resolve()),
    "val": str((SPLIT_DIR / "val.txt").resolve()),
    "test": str((SPLIT_DIR / "test.txt").resolve()),
    "nc": meta["nc"],
    "names": meta["names"],
}

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

data_config


{'path': '/kaggle/input/datasets/matweyisupov/self-driving-card',
 'train': '/kaggle/working/splits/self_driving_car/train.txt',
 'val': '/kaggle/working/splits/self_driving_car/val.txt',
 'test': '/kaggle/working/splits/self_driving_car/test.txt',
 'nc': 11,
 'names': ['biker',
  'car',
  'pedestrian',
  'trafficLight',
  'trafficLight-Green',
  'trafficLight-GreenLeft',
  'trafficLight-Red',
  'trafficLight-RedLeft',
  'trafficLight-Yellow',
  'trafficLight-YellowLeft',
  'truck']}

In [ ]:
model = YOLO(MODEL)

train_args = {
    "data": str(DATA_YAML),
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "project": str(PROJECT),
    "name": NAME,
    "exist_ok": True,
    "seed": SEED,
    "device": DEVICES,
    "deterministic": True,
    "optimizer": "AdamW",
    "lr0": 1e-3,
    "lrf": 1e-2,
    "weight_decay": 5e-4,
    "warmup_epochs": 3,
    "close_mosaic": 10,
    "cos_lr": True,
    "patience": 15,
    "workers": 4,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "translate": 0.1,
    "scale": 0.5,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.0,
    "amp": True,
    "plots": True,
    "save": True,
    "save_period": 5,
    "verbose": True,
}


train_results = model.train(**train_args)


Ultralytics 8.4.26 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/splits/self_driving_car/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_basel

In [ ]:
run_dir = PROJECT / NAME
best_weights = run_dir / "weights" / "best.pt"

best_model = YOLO(str(best_weights)) if best_weights.exists() else model
val_results = best_model.val(data=str(DATA_YAML), split="val")
test_results = best_model.val(data=str(DATA_YAML), split="test")

result = {
    "dataset_root": str(DATASET_ROOT),
    "data_yaml": str(DATA_YAML),
    "run_dir": str(run_dir),
    "best_weights": str(best_weights),
    "train_metrics": getattr(train_results, "results_dict", {}),
    "val_metrics": getattr(val_results, "results_dict", {}),
    "test_metrics": getattr(test_results, "results_dict", {}),
}

(run_dir / "result.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
result
